# 48 — Subword tokenizers (BPE)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/048-subword-tokenizers-bpe.ipynb)

Byte-pair encoding learns by gluing together the most common adjacent symbols. In this notebook you run four merges on a five-word corpus, watch the token count fall, and encode a word that was never in the training data.

In [ ]:
from collections import Counter                             # pair counting
corpus = {"low": 3, "lower": 2}                             # word -> frequency
words = {tuple(w): f for w, f in corpus.items()}            # characters as symbols
def top_pair(words):                                        # most frequent adjacent pair
    counts = Counter()                                      # pair -> count
    for symbols, f in words.items():                        # each word type
        for a, b in zip(symbols, symbols[1:]):              # adjacent pairs only
            counts[(a, b)] += f                             # weight by frequency
    best = max(counts.values())                             # highest count
    return min(p for p, n in counts.items() if n == best)   # lexicographic tie-break
def merge(words, pair):                                     # glue one pair everywhere
    out = {}                                                # new word table
    for symbols, f in words.items():                        # each word type
        merged, i = [], 0                                   # rebuilt symbols
        while i < len(symbols):                             # scan left to right
            if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:  # match
                merged.append(symbols[i] + symbols[i + 1])  # glue the pair
                i += 2                                      # skip both symbols
            else:                                           # no match here
                merged.append(symbols[i])                   # keep the symbol
                i += 1                                      # advance one
        key = tuple(merged)                                 # hashable word
        out[key] = out.get(key, 0) + f                      # accumulate frequency
    return out                                              # merged table
history = []                                                # tokens per merge
for rank in range(1, 5):                                    # four merges
    pair = top_pair(words)                                  # choose the winner
    words = merge(words, pair)                              # apply it
    n = sum(len(s) * f for s, f in words.items())           # total token count
    history.append((rank, n))                               # record the count
    print(rank, pair, n)                                    # show progress

## Read the merge loop

`words` maps a tuple of symbols to its corpus frequency, so `('l', 'o', 'w')` with value 3 means the word appeared three times. `top_pair` counts adjacent pairs weighted by that frequency; `merge` rebuilds every word with the winning pair collapsed into one symbol. After four merges the corpus is `{('low',): 3, ('lower',): 2}`, and the total token count is 5.

The vocabulary grows by one symbol per merge: 256 bytes plus 4 merges gives 260.

In [ ]:
import matplotlib.pyplot as plt                             # plotting library
ranks = [0] + [r for r, _ in history]                       # merge ranks
counts = [19] + [n for _, n in history]                     # token totals
plt.figure(figsize=(6, 3))                                  # wide figure
plt.plot(ranks, counts, marker="o", color="tab:blue")        # compression curve
plt.xlabel("merges applied")                                # x label
plt.ylabel("tokens in the corpus")                           # y label
plt.title("each merge removes symbols")                     # takeaway
plt.show()                                                  # render

## Unseen words

Encoding replays the learned merges in rank order. Take `lowest`: it starts as `l o w e s t`, merge 1 makes `lo w e s t`, and merge 2 makes `low e s t`. Merges 3 and 4 need `e r`, which is absent, so the result is four tokens. The word was never in the corpus, but no unknown token is required.

This is the central difference from a word-level vocabulary: coverage comes from pieces, not from having seen the whole word.

## Your turn

Add `"lowest": 1` to the corpus and rerun. Which merge changes, and does `lowest` become one token? Then write an `encode(word, merges)` function that applies the recorded merges in rank order, and check it reproduces the four-token result. Finally, compute the average tokens per word before and after the merges.